# S8 - Streaming estructurado sobre sensor real (Proyecto Sello)

**Equipo:** BIG IMPACT (10) · **Estudiante:** Nilver Salcca Aquino
**Fuente:** ESP32 + DHT22 real → MQTT → bridge → Kafka (`clima-juliaca-eventos`)

## 3.1 Parámetros del proyecto

In [1]:
TOPIC = "clima-juliaca-eventos"
BOOTSTRAP = "kafka:9092"
ARTIFACTS_DIR = "/opt/sesion8/artifacts"
print("Topic:", TOPIC)
print("Salida:", ARTIFACTS_DIR)

Topic: clima-juliaca-eventos
Salida: /opt/sesion8/artifacts


## 3.2 Crear la SparkSession, con el conector de Kafka

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion8-streaming-clima-juliaca")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7e52cf9b-8cc1-4198-9fec-0ea8aac59049;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

## 3.3 Verificar el topic con una lectura batch

In [3]:
lote = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)
print("mensajes en el topic:", lote.count())
lote.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "partition", "offset").orderBy("offset", ascending=False).show(3, truncate=False)

mensajes en el topic: 276
+--------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|key                 |value                                                                                                                                                                     |partition|offset|
+--------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|esp32-juliaca-nilver|{"tipoEvento":"clima.lectura","sensorId":"esp32-juliaca-nilver","temperatura":19.7,"humedad":9.3,"presion":1008.2,"origen":"esp32-real-juliaca","timestamp":1791418454000}|0        |276   |
|esp32-juliaca-nilver|{"tipoEvento":"clima.lectura","sensorId":"esp32-juliaca-nilver","temperatura":19.7,"humedad":9.3,"presion":1

## 3.4 Leer el topic en modo streaming

In [4]:
crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)
print("es un stream:", crudo.isStreaming)
crudo.printSchema()

es un stream: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## 3.5 Parsear el JSON con un esquema explícito (contrato clima.lectura)

In [5]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temperatura", DoubleType())
    .add("humedad", DoubleType())
    .add("presion", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

eventos = (
    crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
)
eventos.printSchema()

root
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)



## 3.6 Primera consulta: ver llegar las lecturas del ESP32


In [6]:
consulta = (
    eventos.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(20)
consulta.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+----------+--------+-----------+-------+-------+------+---------+---+
|tipoEvento|sensorId|temperatura|humedad|presion|origen|timestamp|ts |
+----------+--------+-----------+-------+-------+------+---------+---+
+----------+--------+-----------+-------+-------+------+---------+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+--------------------+-----------+-------+-------+------------------+-------------+-------------------+
|tipoEvento   |sensorId            |temperatura|humedad|presion|origen            |timestamp    |ts                 |
+-------------+--------------------+-----------+-------+-------+------------------+-------------+-------------------+
|clima.lectura|esp32-juliaca-nilver|19.4       |8.1    |1008.7 |esp32-real-juliaca|1791418779000|2026-10-08 00:19:39|
+-------------+--------------------+-----------+-

## 3.7 Agregar por ventana de tiempo, sin watermark (estado sin límite)

In [7]:
from pyspark.sql.functions import window, avg, count, min as spark_min

agregado_sin_limite = (
    eventos.groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(
        avg("temperatura").alias("temp_prom"),
        spark_min("temperatura").alias("temp_min"),
        count("*").alias("n"),
    )
)

consulta = (
    agregado_sin_limite.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(30)
consulta.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+---------+--------+---+
|window|sensorId|temp_prom|temp_min|n  |
+------+--------+---------+--------+---+
+------+--------+---------+--------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+--------------------+---------+--------+---+
|window                                    |sensorId            |temp_prom|temp_min|n  |
+------------------------------------------+--------------------+---------+--------+---+
|{2026-10-08 00:20:50, 2026-10-08 00:21:00}|esp32-juliaca-nilver|19.5     |19.5    |1  |
+------------------------------------------+--------------------+---------+--------+---+

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+--------------------+---------+--------+---+
|window                                    |sensorId            |temp_prom|temp_min|n  |
+------------------------------------------+--------------------+---------+--------+---+
|{2026-10-08 00:20:50, 2026-10-08 00:21:00}|esp32-juliaca-nilver|19.5     |19.5    |2  |
+---

## 3.8 Watermarking: acotar cuánto se espera por datos tardíos

**Margen elegido: 10 segundos.** El ESP32 publica cada 5 s y la latencia medida hasta Kafka es de ~1.5 s, así que 10 s equivale a tolerar un par de lecturas retrasadas o perdidas (por ejemplo una reconexión breve del WiFi). **Se gana:** estado acotado en memoria y ventanas que se cierran rápido. **Se pierde:** cualquier lectura que llegue con más de 10 s de retraso se descarta.

In [8]:
agregado_con_watermark = (
    eventos
    .withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(
        avg("temperatura").alias("temp_prom"),
        spark_min("temperatura").alias("temp_min"),
        count("*").alias("n"),
    )
)

consulta = (
    agregado_con_watermark.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-clima-chk-watermark")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(40)
consulta.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+---------+--------+---+
|window|sensorId|temp_prom|temp_min|n  |
+------+--------+---------+--------+---+
+------+--------+---------+--------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+--------------------+---------+--------+---+
|window                                    |sensorId            |temp_prom|temp_min|n  |
+------------------------------------------+--------------------+---------+--------+---+
|{2026-10-08 00:23:00, 2026-10-08 00:23:10}|esp32-juliaca-nilver|19.7     |19.7    |1  |
+------------------------------------------+--------------------+---------+--------+---+

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+--------------------+---------+--------+---+
|window                                    |sensorId            |temp_prom|temp_min|n  |
+------------------------------------------+--------------------+---------+--------+---+
|{2026-10-08 00:23:10, 2026-10-08 00:23:20}|esp32-juliaca-nilver|19.7     |19.7    |1  |
+---

## 3.9 Checkpointing: qué guarda y qué pasa al reiniciar

In [9]:
import os

CHK = "/tmp/s08-clima-chk-watermark"
for carpeta in sorted(os.listdir(CHK)):
    print(carpeta)

.metadata.crc
commits
metadata
offsets
sources
state


In [10]:
for sub in ["offsets", "commits", "state"]:
    print(sub, "->", sorted(os.listdir(f"{CHK}/{sub}"))[:10])

numeros = sorted((a for a in os.listdir(f"{CHK}/offsets") if a.isdigit()), key=int)
ultimo = numeros[-1]
print(f"\nContenido de offsets/{ultimo}:")
with open(f"{CHK}/offsets/{ultimo}") as f:
    print(f.read())

offsets -> ['.0.crc', '.1.crc', '.2.crc', '.3.crc', '.4.crc', '.5.crc', '.6.crc', '.7.crc', '0', '1']
commits -> ['.0.crc', '.1.crc', '.2.crc', '.3.crc', '.4.crc', '.5.crc', '.6.crc', '.7.crc', '0', '1']
state -> ['0']

Contenido de offsets/7:
v1
{"batchWatermarkMs":1791419004000,"batchTimestampMs":1791419025009,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.stateStore.rocksdb.mergeOperatorVersion":"2","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.queryEvolution.enableSourceEvolution":"false","spark.sql.streaming.stateStore.encodingFormat":"unsaferow","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.stateStore.rowChecksum.enabled":"false","spark.sql.streaming.aggregation.stateFormatVersion"

## 3.10 Un dato tardío y el watermark descartándolo

In [11]:
import threading
import time
from pyspark.sql.functions import to_json, struct

COLUMNAS = ["tipoEvento", "sensorId", "temperatura", "humedad", "presion", "origen", "timestamp"]

def publicar_evento(sensor_id, origen, segundos_de_espera, segundos_en_el_pasado, veces=1):
    time.sleep(segundos_de_espera)
    ts_ms = int(time.time() * 1000) - segundos_en_el_pasado * 1000
    fila = [("clima.lectura", sensor_id, -3.0, 40.0, 1013.0, origen, ts_ms)]
    df = spark.createDataFrame(fila, COLUMNAS)
    salida = df.select(df.sensorId.cast("string").alias("key"), to_json(struct(*COLUMNAS)).alias("value"))
    for _ in range(veces):
        (salida.write.format("kafka")
            .option("kafka.bootstrap.servers", BOOTSTRAP)
            .option("topic", TOPIC)
            .save())
    print(f">>> publicado {sensor_id} x{veces}, {segundos_en_el_pasado}s en el pasado (timestamp={ts_ms})")

In [12]:
hilo = threading.Thread(target=publicar_evento, args=("esp32-tardio", "prueba-watermark", 15, 45), daemon=True)
hilo.start()

agregado = (
    eventos
    .withWatermark("ts", "5 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(count("*").alias("n"))
)

consulta = (
    agregado.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-clima-chk-tardio")
    .trigger(processingTime="4 seconds")
    .start()
)
consulta.awaitTermination(45)
consulta.stop()
print("Busca 'esp32-tardio' en las tablas de arriba: si no aparece ninguna fila, el watermark lo descarto.")

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+---+
|window|sensorId|n  |
+------+--------+---+
+------+--------+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+--------------------+---+
|window                                    |sensorId            |n  |
+------------------------------------------+--------------------+---+
|{2026-10-08 00:26:20, 2026-10-08 00:26:30}|esp32-juliaca-nilver|1  |
+------------------------------------------+--------------------+---+

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+--------------------+---+
|window                                    |sensorId            |n  |
+------------------------------------------+--------------------+---+
|{2026-10-08 00:26:30, 2026-10-08 00:26:40}|esp32-juliaca-n

>>> publicado esp32-tardio x1, 45s en el pasado (timestamp=1791419158286)
-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+--------------------+---+
|window                                    |sensorId            |n  |
+------------------------------------------+--------------------+---+
|{2026-10-08 00:26:30, 2026-10-08 00:26:40}|esp32-juliaca-nilver|2  |
+------------------------------------------+--------------------+---+

-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+--------------------+---+
|window                                    |sensorId            |n  |
+------------------------------------------+--------------------+---+
|{2026-10-08 00:26:40, 2026-10-08 00:26:50}|esp32-juliaca-nilver|1  |
+------------------------------------------+--------------------+---+

------------------------------

## 3.11 Deduplicación acotada por watermark

In [13]:
deduplicado = (
    eventos
    .withWatermark("ts", "30 seconds")
    .dropDuplicates(["sensorId", "timestamp"])
)

consulta = (
    deduplicado.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-clima-chk-dedup")
    .trigger(processingTime="3 seconds")
    .start()
)

time.sleep(6)
publicar_evento("esp32-duplicado", "prueba-dedup", 0, 0, veces=2)

consulta.awaitTermination(25)
consulta.stop()
print("Busca 'esp32-duplicado' arriba: debe aparecer una sola vez, aunque se publico dos veces.")

-------------------------------------------
Batch: 0
-------------------------------------------
+----------+--------+-----------+-------+-------+------+---------+---+
|tipoEvento|sensorId|temperatura|humedad|presion|origen|timestamp|ts |
+----------+--------+-----------+-------+-------+------+---------+---+
+----------+--------+-----------+-------+-------+------+---------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+--------------------+-----------+-------+-------+------------------+-------------+-------------------+
|tipoEvento   |sensorId            |temperatura|humedad|presion|origen            |timestamp    |ts                 |
+-------------+--------------------+-----------+-------+-------+------------------+-------------+-------------------+
|clima.lectura|esp32-juliaca-nilver|20.8       |8.4    |1008.1 |esp32-real-juliaca|1791419294000|2026-10-08 00:28:14|
+-------------+--------------------+-----------+-------+-------+------------------+-------------+-------------------+

>>> publicado esp32-duplicado x2, 0s en el pasado (timestamp=1791419298037)
-------------------------------------------
Batch: 2
-------------------------------------------
+-------------+--------------------+-----------+-------+-------+------------------+-------------+-----------------------+
|tipoEvento   |se

## 3.12 Intervalo de disparo: el costo de decidir cada cuánto procesar

In [14]:
def contar_microlotes(intervalo, segundos_totales):
    nombre = f"conteo_{intervalo.replace(' ', '_')}"
    consulta = (
        eventos.writeStream
        .outputMode("append")
        .format("memory")
        .queryName(nombre)
        .trigger(processingTime=intervalo)
        .start()
    )
    consulta.awaitTermination(segundos_totales)
    consulta.stop()
    ultimo = consulta.lastProgress["batchId"] if consulta.lastProgress else None
    filas = spark.table(nombre).count()
    return ultimo, filas

lote_rapido, filas_rapido = contar_microlotes("1 second", 30)
lote_lento, filas_lento = contar_microlotes("15 seconds", 30)

print(f"Disparo cada 1 segundo : ultimo batchId = {lote_rapido}, filas procesadas = {filas_rapido}")
print(f"Disparo cada 15 segundos: ultimo batchId = {lote_lento}, filas procesadas = {filas_lento}")

Disparo cada 1 segundo : ultimo batchId = 6, filas procesadas = 6
Disparo cada 15 segundos: ultimo batchId = 2, filas procesadas = 5


## 3.13 Escribir la salida a Parquet, con checkpoint

In [15]:
ARTIFACTS = f"{ARTIFACTS_DIR}/clima_parquet"
CHECKPOINT_PARQUET = f"{ARTIFACTS_DIR}/chk_clima_parquet"

def correr_parquet(segundos):
    consulta = (
        eventos.writeStream
        .outputMode("append")
        .format("parquet")
        .option("path", ARTIFACTS)
        .option("checkpointLocation", CHECKPOINT_PARQUET)
        .trigger(processingTime="5 seconds")
        .start()
    )
    consulta.awaitTermination(segundos)
    consulta.stop()
    return spark.read.parquet(ARTIFACTS).count()

total_corrida_1 = correr_parquet(30)
print("Corrida 1 -> filas guardadas en Parquet:", total_corrida_1)
spark.read.parquet(ARTIFACTS).select("sensorId", "temperatura", "humedad", "ts").orderBy("ts", ascending=False).show(5, truncate=False)

Corrida 1 -> filas guardadas en Parquet: 5
+--------------------+-----------+-------+-------------------+
|sensorId            |temperatura|humedad|ts                 |
+--------------------+-----------+-------+-------------------+
|esp32-juliaca-nilver|20.5       |6.9    |2026-10-08 00:32:49|
|esp32-juliaca-nilver|20.5       |6.9    |2026-10-08 00:32:44|
|esp32-juliaca-nilver|20.5       |6.9    |2026-10-08 00:32:39|
|esp32-juliaca-nilver|20.5       |6.9    |2026-10-08 00:32:34|
|esp32-juliaca-nilver|20.5       |6.9    |2026-10-08 00:32:29|
+--------------------+-----------+-------+-------------------+



**Segunda corrida:** se ejecuta la misma consulta, con el mismo checkpoint. El total de filas debe crecer, no reiniciarse.

In [16]:
total_corrida_2 = correr_parquet(30)
archivos = [f for f in os.listdir(ARTIFACTS) if f.endswith(".parquet")]

print("Corrida 1 -> filas:", total_corrida_1)
print("Corrida 2 -> filas:", total_corrida_2)
print("Acumula, no sobreescribe:", total_corrida_2 > total_corrida_1)
print("Archivos .parquet en disco:", len(archivos))

26/10/08 00:34:56 ERROR FileFormatWriter: Aborting job c9feb139-861a-46bb-ab54-a782a1af8f69.
java.io.InterruptedIOException: java.lang.InterruptedException
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1074)
	at org.apache.hadoop.util.Shell.run(Shell.java:960)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1285)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1380)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1362)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem$LocalFSFileOutputStream.<init>(RawLocalFileSystem.java:517)
	at org.apache.hadoop.fs.RawLocalFileSystem.createOutputStreamWithMode(RawLocalFileSystem.java:640)
	at org.apache.hadoop.fs.RawLocalFileSystem.create(RawLocalFileSystem.java:629)
	at org.apache.hadoop.fs.RawLocalFileSystem.create(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.FileSystem.primitiveCreate(FileSystem.java:1357)
	

Corrida 1 -> filas: 5
Corrida 2 -> filas: 28
Acumula, no sobreescribe: True
Archivos .parquet en disco: 11


## Error provocado: consulta a Parquet sin checkpointLocation

In [17]:
try:
    q = (
        eventos.writeStream
        .outputMode("append")
        .format("parquet")
        .option("path", f"{ARTIFACTS_DIR}/prueba_sin_checkpoint")
        .start()
    )
    q.stop()
    print("La consulta arranco (sin error)")
except Exception as e:
    print("Tipo de error:", type(e).__name__)
    print(str(e)[:500])

Tipo de error: AnalysisException
checkpointLocation must be specified either through option("checkpointLocation", ...) or SparkSession.conf.set("spark.sql.streaming.checkpointLocation", ...).
